 Imports and setup

In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import joblib

from src.models.features import build_dataset, time_series_split
from src.models.xgboost_model import (
    get_xgb_params,
    get_prediction_explanation
)
from src.models.explainer import (
    generate_explanation,
    generate_indicator_explanation,
    batch_explain_portfolio,
    FEATURE_LABELS
)
from src.data.database import load_prices, get_connection
from sqlalchemy import text
import shap
import xgboost as xgb

TICKER    = "RELIANCE.NS"
WATCHLIST = [
    "RELIANCE.NS", "TCS.NS", "INFY.NS",
    "HDFCBANK.NS", "WIPRO.NS"
]
HORIZON   = 10

print("Imports done.")

C:\Users\devgi\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Imports done.


Load model and get today's prediction

In [2]:
# Load the trained model
xgb_model = joblib.load("../models/xgboost_model.pkl")

# Check how many features the model expects
expected_features = xgb_model.get_booster().feature_names
print(f"Model expects {len(expected_features)} features:")
print(expected_features)

# Build dataset with the correct feature set
# If model has sentiment features, use load_features_with_sentiment
from src.data.database import load_features_with_sentiment
from src.models.features import time_series_split

HORIZON = 10

def build_dataset_with_sentiment(ticker, horizon=10):
    """Build dataset matching the Week 7 trained model's feature set."""
    from src.data.database import load_features_with_sentiment
    import pandas as pd

    df = load_features_with_sentiment(ticker)

    future_close  = df["close"].shift(-horizon)
    future_return = (future_close - df["close"]) / df["close"]
    y = (future_return > 0).astype(int)

    df["price_vs_sma20"] = df["close"] / df["sma_20"] - 1
    df["price_vs_sma50"] = df["close"] / df["sma_50"] - 1
    df["rsi_normalized"] = (df["rsi"] - 50) / 50

    feature_cols = [
        "rsi", "macd", "macd_signal", "macd_hist",
        "bb_pct", "bb_width", "sma_20", "sma_50",
        "ema_20", "volume_ratio",
        "price_vs_sma20", "price_vs_sma50", "rsi_normalized",
        "sentiment_1d", "sentiment_3d",
        "sentiment_7d", "sentiment_momentum"
    ]

    X = df[feature_cols].copy()
    X = X.iloc[:-horizon]
    y = y.iloc[:-horizon]

    valid = X.notna().all(axis=1) & y.notna()
    X, y  = X[valid], y[valid]

    return X, y

X, y = build_dataset_with_sentiment(TICKER, HORIZON)
X_train, X_test, y_train, y_test = time_series_split(X, y, 0.2)

# Rebuild SHAP explainer
explainer = shap.TreeExplainer(xgb_model)

# Get today's prediction
today_row  = X_test.iloc[[-1]]
today_date = X_test.index[-1].strftime("%Y-%m-%d")

explanation_data = get_prediction_explanation(
    xgb_model, explainer, today_row
)

print(f"\nDate        : {today_date}")
print(f"P(UP)       : {explanation_data['probability_up']:.1%}")
print(f"P(DOWN)     : {explanation_data['probability_down']:.1%}")
print(f"Confidence  : {explanation_data['confidence'].upper()}")
print(f"\nTop features:")
for f in explanation_data["top_features"]:
    sign = "+" if f["shap_value"] > 0 else ""
    print(f"  {f['feature']:<22} {f['direction']:<10} "
          f"{sign}{f['shap_value']:.4f}")

Model expects 17 features:
['rsi', 'macd', 'macd_signal', 'macd_hist', 'bb_pct', 'bb_width', 'sma_20', 'sma_50', 'ema_20', 'volume_ratio', 'price_vs_sma20', 'price_vs_sma50', 'rsi_normalized', 'sentiment_1d', 'sentiment_3d', 'sentiment_7d', 'sentiment_momentum']
Train: 352 days (2024-09-12 → 2026-02-10)
Test:  88 days  (2026-02-11 → 2026-06-19)

Date        : 2026-06-19
P(UP)       : 49.0%
P(DOWN)     : 51.0%
Confidence  : LOW

Top features:
  macd_hist              bearish    -1.5339
  macd_signal            bullish    +1.1878
  price_vs_sma50         bearish    -0.5679
  volume_ratio           bullish    +0.5358
  sma_20                 bearish    -0.3837


Get company name from DB

In [3]:
# Load company metadata saved
with get_connection() as conn:
    result = conn.execute(text("""
        SELECT company_name FROM stock_metadata
        WHERE ticker = :ticker
    """), {"ticker": TICKER})
    row = result.fetchone()

company_name = row[0] if row else TICKER.replace(".NS", "")
print(f"Company: {company_name}")

Company: RELIANCE


Generate plain-English explanation

In [4]:
explanation = generate_explanation(
    ticker       = TICKER,
    company_name = company_name,
    date         = today_date,
    prob_up      = explanation_data["probability_up"],
    confidence   = explanation_data["confidence"],
    top_features = explanation_data["top_features"],
    horizon      = HORIZON
)

print(f"\n{explanation}")
print("\n" + "─" * 55)
print(f"Word count: {len(explanation.split())} words")


RELIANCE shows a 49.0% probability of a price fall over the next 10 trading days. The strongest signal comes from the MACD momentum histogram, which is currently bearish. Model confidence is low. Remember: this is a probability estimate, not a guarantee.

───────────────────────────────────────────────────────
Word count: 41 words


Test indicator explanation (tooltip feature)

In [5]:
print("Testing indicator explanations (frontend tooltips)...")
print("─" * 55)

# Get current indicator values
last_row = X_test.iloc[-1]

indicators_to_explain = [
    ("RSI", float(last_row["rsi"])),
    ("MACD histogram", float(last_row["macd_hist"])),
    ("Bollinger %B", float(last_row["bb_pct"])),
]

for name, value in indicators_to_explain:
    print(f"\n{name} = {value:.2f}")
    tooltip = generate_indicator_explanation(name, value, TICKER)
    print(f"  → {tooltip}")

Testing indicator explanations (frontend tooltips)...
───────────────────────────────────────────────────────

RSI = 46.80
  → An RSI of 46.80 for Reliance means the stock is currently in a "neutral" zone. It is neither overbought (too expensive) nor oversold (too cheap). It suggests the stock is experiencing moderate momentum, with selling pressure slightly outweighing buying pressure, but without a strong trend in either direction. It is essentially a "wait and see" state.

MACD histogram = 6.14
  → A MACD histogram value of 6.14 means the "fast" moving average is currently 6.14 points above the "slow" moving average. A positive number indicates that the stock's recent price momentum is stronger than its longer-term trend. The higher the value, the stronger the upward momentum currently driving Reliance’s price.

Bollinger %B = 0.50
  → A Bollinger %B value of 0.50 means the current price of Reliance is exactly in the middle of its Bollinger Bands. Since the bands represent the typic

Portfolio-level explanations

In [6]:
print("Generating portfolio-level explanations...")
print("─" * 55)

# Build predictions for all watchlist tickers
portfolio_preds = []

for ticker in WATCHLIST:
    try:
        X_t, y_t, _ = build_dataset(ticker, horizon=HORIZON)
        X_tr, X_te, y_tr, y_te = time_series_split(X_t, y_t, 0.2)

        # Train fresh model for each ticker
        model_t = xgb.XGBClassifier(**get_xgb_params())
        model_t.fit(X_tr, y_tr, verbose=False)

        expl_t    = shap.TreeExplainer(model_t)
        today_t   = X_te.iloc[[-1]]
        pred_t    = get_prediction_explanation(
            model_t, expl_t, today_t
        )

        # Get company name
        with get_connection() as conn:
            result = conn.execute(text("""
                SELECT company_name FROM stock_metadata
                WHERE ticker = :ticker
            """), {"ticker": ticker})
            row = result.fetchone()
        name = row[0] if row else ticker

        portfolio_preds.append({
            "ticker"      : ticker,
            "company_name": name,
            "date"        : X_te.index[-1].strftime("%Y-%m-%d"),
            "prob_up"     : pred_t["probability_up"],
            "confidence"  : pred_t["confidence"],
            "top_features": pred_t["top_features"],
        })
        print(f"  ✓ {ticker}: P(UP)={pred_t['probability_up']:.1%} "
              f"({pred_t['confidence']})")

    except Exception as e:
        print(f"  ✗ {ticker}: {e}")

print(f"\nBuilt predictions for {len(portfolio_preds)} tickers.")

Generating portfolio-level explanations...
───────────────────────────────────────────────────────
Dataset built for RELIANCE.NS
  Total samples : 440
  Features      : 13
  UP days       : 206 (46.8%)
  DOWN days     : 234 (53.2%)
  Date range    : 2024-09-12 → 2026-06-19
Train: 352 days (2024-09-12 → 2026-02-10)
Test:  88 days  (2026-02-11 → 2026-06-19)
  ✓ RELIANCE.NS: P(UP)=30.2% (medium)
Dataset built for TCS.NS
  Total samples : 440
  Features      : 13
  UP days       : 179 (40.7%)
  DOWN days     : 261 (59.3%)
  Date range    : 2024-09-12 → 2026-06-19
Train: 352 days (2024-09-12 → 2026-02-10)
Test:  88 days  (2026-02-11 → 2026-06-19)
  ✓ TCS.NS: P(UP)=98.0% (high)
Dataset built for INFY.NS
  Total samples : 440
  Features      : 13
  UP days       : 220 (50.0%)
  DOWN days     : 220 (50.0%)
  Date range    : 2024-09-12 → 2026-06-19
Train: 352 days (2024-09-12 → 2026-02-10)
Test:  88 days  (2026-02-11 → 2026-06-19)
  ✓ INFY.NS: P(UP)=93.4% (high)
Dataset built for HDFCBANK.NS
  

Generate all portfolio explanations

In [7]:
print("Generating explanations for all tickers...")
print("(One API call per ticker — takes ~10 seconds)\n")

portfolio_with_explanations = batch_explain_portfolio(
    portfolio_preds
)

print("\n" + "═"*60)
print("PORTFOLIO SIGNAL SUMMARY")
print("═"*60)

for pred in portfolio_with_explanations:
    print(f"\n{'─'*60}")
    print(f"{pred['company_name']} ({pred['ticker']})")
    print(f"P(UP in 10d) = {pred['prob_up']:.1%}  |  "
          f"Confidence: {pred['confidence'].upper()}")
    print(f"\n{pred['explanation']}")

print(f"\n{'═'*60}")
print("This is the exact output the /explain API endpoint serves.")
print("The frontend renders this in the explanation panel.")

Generating explanations for all tickers...
(One API call per ticker — takes ~10 seconds)


════════════════════════════════════════════════════════════
PORTFOLIO SIGNAL SUMMARY
════════════════════════════════════════════════════════════

────────────────────────────────────────────────────────────
RELIANCE.NS (RELIANCE.NS)
P(UP in 10d) = 30.2%  |  Confidence: MEDIUM

Our model shows a 69.8% probability of a downward trend for Reliance with medium confidence. The primary signals are bearish, specifically the MACD momentum histogram [a tool measuring momentum acceleration] and the price sitting below its 50-day average. While trading volume is currently high, the short-term trend remains weak. Please remember that market predictions are probabilistic and even high-confidence signals can be incorrect.

────────────────────────────────────────────────────────────
TCS.NS (TCS.NS)
P(UP in 10d) = 98.0%  |  Confidence: HIGH

TCS shows a 98% probability of an upward trend with high confidence.

 Inspect the prompt structure

In [8]:
from src.models.explainer import build_explanation_prompt, SYSTEM_PROMPT

print("SYSTEM PROMPT:")
print("─" * 55)
print(SYSTEM_PROMPT)

print("\n\nUSER PROMPT (for today's RELIANCE prediction):")
print("─" * 55)
user_prompt = build_explanation_prompt(
    ticker       = TICKER,
    company_name = company_name,
    date         = today_date,
    prob_up      = explanation_data["probability_up"],
    confidence   = explanation_data["confidence"],
    top_features = explanation_data["top_features"],
    horizon      = HORIZON
)
print(user_prompt)

SYSTEM PROMPT:
───────────────────────────────────────────────────────
You are InvestIQ's explanation engine. You translate machine
learning model outputs into plain English for beginner investors.

Your tone is friendly, clear, and honest. You never recommend
buying or selling — you only explain what the signals suggest
and how confident the model is.

You always remind users that stock market predictions are
probabilistic — even a high-confidence signal can be wrong.

You explain technical terms in brackets when you must use them.
You keep explanations concise — under 150 words.


USER PROMPT (for today's RELIANCE prediction):
───────────────────────────────────────────────────────
You are InvestIQ's explanation engine. You translate machine
learning model outputs into plain English for beginner investors.

Your tone is friendly, clear, and honest. You never recommend
buying or selling — you only explain what the signals suggest
and how confident the model is.

You always remind user

Cache Simulation

In [9]:

import hashlib
import json

def make_cache_key(ticker, date, prob_up, top_features):
    """
    Cache key is a hash of the inputs.
    Same inputs → same key → cache hit → no API call needed.
    Different inputs (new prediction) → cache miss → API call.
    """
    content = json.dumps({
        "ticker"    : ticker,
        "date"      : date,
        "prob_up"   : round(prob_up, 3),
        "features"  : [f["feature"] for f in top_features[:3]]
    }, sort_keys=True)
    return f"explain:{hashlib.md5(content.encode()).hexdigest()}"

cache_key = make_cache_key(
    TICKER, today_date,
    explanation_data["probability_up"],
    explanation_data["top_features"]
)

print(f"Cache key for today's RELIANCE prediction:")
print(f"  {cache_key}")
print(f"\nIn production (Week 11):")
print(f"  1. Check Redis: GET {cache_key}")
print(f"  2. Cache hit  → return stored explanation instantly")
print(f"  3. Cache miss → call GEMINI API → store with 15min TTL")
print(f"\nThis prevents charging for the same explanation twice")
print(f"when multiple users request the same ticker on the same day.")

Cache key for today's RELIANCE prediction:
  explain:63aa20f8137839ea6a402b54054102ee

In production (Week 11):
  1. Check Redis: GET explain:63aa20f8137839ea6a402b54054102ee
  2. Cache hit  → return stored explanation instantly
  3. Cache miss → call GEMINI API → store with 15min TTL

This prevents charging for the same explanation twice
when multiple users request the same ticker on the same day.
